# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR⁲) Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and contains multiple record sets, fields, and columns. All dataset structural elements (record sets, fields, etc.) are referenced by their `@id` values for precision and reproducibility.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The FAIR⁲ schema describes tabular clinical-pathological data. Let's enumerate all available record sets and their fields, referencing each by its `@id`. 

In [ ]:
# List all record sets and their fields
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    Name: {getattr(rs, 'name', None)}")
    print(f"    Fields:")
    for field in rs.fields:
        print(f"      Field @id: {field.id} (name: {getattr(field, 'name', None)})")
    print()
# Save list of RecordSet @ids for later extraction
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction

Load data from each available record set into a DataFrame for analysis. All record sets and field names are referenced by their Croissant `@id`.

In [ ]:
# Extract data per record set, using @id as key
dataframes = {}

for record_set_id in record_set_ids:
    # Retrieve records for each record set by @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'RecordSet @id: {record_set_id} --> shape: {df.shape}')

# For demonstration, display columns and a preview from the primary record set (typically only one)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in RecordSet @id: {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping/categorizing data.

Be sure to reference all fields by their exact Croissant `@id`. Let's perform some basic operations on a selected numeric field (e.g. patient age or interval months).

In [ ]:
# Pick main table (record set) to analyze, and inspect its columns
df = dataframes[main_record_set_id]
print(f"Columns in {main_record_set_id}:")
print(df.columns.tolist())

# Try to auto-select a numeric field by inspecting column datatypes, else default to a likely one
numeric_field_id = None
potential_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if potential_numeric_fields:
    numeric_field_id = potential_numeric_fields[0]
else:
    # Try common clinical terms (by @id): e.g. 'age', 'interval_months', etc.
    for candidate in ['age', 'interval_months', 'schema:age', 'schema:IntervalMonths', 'cr:age']:
        if candidate in df.columns:
            numeric_field_id = candidate
            break
    if numeric_field_id is None:
        # Use any column with numbers in first row
        for col in df.columns:
            if df[col].dtype == object and pd.to_numeric(df[col], errors='coerce').notna().all():
                numeric_field_id = col
                break

if numeric_field_id is None:
    raise ValueError("No suitable numeric field found for EDA.")
print(f"\nSelected numeric field by @id: {numeric_field_id}")

# Set threshold for filtering
threshold = df[numeric_field_id].mean()  # Use mean as an example threshold

# Filter records where numeric_field > threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the selected numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Optional: Group by a categorical field, such as 'sex', 'MSI_status', or similar
group_field = None
possible_group_cols = [
    col for col in df.columns if \
    col.lower() in ['sex', 'gender', 'msi_status', 'comorbidity', 'cr:sex', 'cr:MSI_status'] 
    or df[col].dtype == 'object' and df[col].nunique() < 10
]
if possible_group_cols:
    group_field = possible_group_cols[0]
    print(f"\nGrouping by {group_field}:")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean').join(
        filtered_df.groupby(group_field)[numeric_field_id].std().to_frame('std'), how='outer')
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to the chosen group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of original numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True, color='royalblue')
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.title(f'Distribution of {numeric_field_id}')
plt.tight_layout()
plt.show()

# If group_field is available, show boxplot by group
if group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x=group_field, y=numeric_field_id, showmeans=True)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

We demonstrated how to use the Croissant schema via `mlcroissant` to:
- Discover record sets and field structure (by `@id`)
- Load all data into Pandas DataFrames
- Perform basic EDA: filtering, normalization, and grouping, referencing all items by their Croissant `@id`
- Visualize key distributions and group differences

All steps were referenced consistently to the dataset's Croissant semantic structure, supporting reproducible clinical-omics research. Continue to adapt this notebook for more advanced modeling or study using the full suite of Croissant features.